# 梯度消失与爆炸：逐层 Hook、初始化与残差修正

**面试问题：深层网络训练不动或 Loss 变 NaN 时，怎样定位梯度消失/爆炸？**

## 回答主线

1. 先记录每层激活均值/方差、梯度范数和参数更新比，而不是只看最终 Loss。
2. Sigmoid 在饱和区导数很小，深层连乘会造成梯度消失。
3. 过大的权重谱半径和激活会造成梯度爆炸甚至 Inf/NaN。
4. Xavier/He 初始化要匹配激活函数，残差连接提供较短梯度路径。
5. Gradient Clipping 只能限制爆炸，不能修复 NaN 来源或梯度消失。
6. 混合精度还要配合 Loss Scaling、非有限值检测和跳步策略。

## 真实案例

用 16 条二维客服风险样本训练一个 10 层、宽度 16 的二分类 MLP。我们分别构造小增益 Sigmoid 连乘、过大 ReLU 和 He 初始化残差网络，手写 `nn.Module` 与 forward，保留每层激活梯度并打印从输出到输入的范数；最后复现“先产生 NaN 再 clipping”无效。使用可读的离线教学数据解释机制，指标不能外推为线上收益。

### 输入预览：十六条二维风险样本

In [1]:
import torch  # 导入 PyTorch 以构建和反向传播深层网络。

torch.manual_seed(6)  # 固定初始化和数据。
features = torch.tensor([[-2.0, -1.0], [-1.8, -0.5], [-1.5, 0.2], [-1.2, -0.8], [-0.9, 0.4], [-0.5, -0.2], [-0.2, 0.1], [0.0, -0.3], [0.2, 0.4], [0.5, 0.2], [0.8, -0.1], [1.0, 0.6], [1.2, 0.3], [1.5, 0.8], [1.8, 0.5], [2.0, 1.0]], dtype=torch.float32)  # 构造从低风险到高风险的二维特征。
labels = ((features[:, 0] + 0.5 * features[:, 1]) > 0.4).float().unsqueeze(1)  # 根据可解释线性边界生成标签。
print("index  feature        label")  # 输出数据表头。
for index, (feature, label) in enumerate(zip(features, labels)):  # 逐样本展示输入和标签。
    print(f"{index:>3}   {feature.tolist()}  {int(label.item())}")  # 展示正负样本分布。
print(f"正例={int(labels.sum())}，负例={len(labels) - int(labels.sum())}")  # 汇总类别。

index  feature        label
  0   [-2.0, -1.0]  0
  1   [-1.7999999523162842, -0.5]  0
  2   [-1.5, 0.20000000298023224]  0
  3   [-1.2000000476837158, -0.800000011920929]  0
  4   [-0.8999999761581421, 0.4000000059604645]  0
  5   [-0.5, -0.20000000298023224]  0
  6   [-0.20000000298023224, 0.10000000149011612]  0
  7   [0.0, -0.30000001192092896]  0
  8   [0.20000000298023224, 0.4000000059604645]  0
  9   [0.5, 0.20000000298023224]  1
 10   [0.800000011920929, -0.10000000149011612]  1
 11   [1.0, 0.6000000238418579]  1
 12   [1.2000000476837158, 0.30000001192092896]  1
 13   [1.5, 0.800000011920929]  1
 14   [1.7999999523162842, 0.5]  1
 15   [2.0, 1.0]  1
正例=7，负例=9


## Baseline 基线：10 层 Sigmoid 导数与小增益连乘导致梯度消失

In [2]:
class DeepMLP(torch.nn.Module):  # 定义可选择激活、初始化和残差的深层网络。
    def __init__(self, activation, weight_scale, residual=False):  # 初始化十层隐藏变换和输出头。
        super().__init__()  # 初始化 Module 基类。
        self.input_layer = torch.nn.Linear(2, 16)  # 将二维输入映射到十六维。
        self.hidden_layers = torch.nn.ModuleList([torch.nn.Linear(16, 16) for _ in range(9)])  # 创建九层隐藏线性层。
        self.output_layer = torch.nn.Linear(16, 1)  # 创建二分类输出头。
        self.activation = activation  # 保存激活函数名称。
        self.residual = residual  # 保存是否使用残差连接。
        for layer in [self.input_layer, *self.hidden_layers, self.output_layer]:  # 逐线性层设置受控初始化。
            torch.nn.init.normal_(layer.weight, mean=0.0, std=weight_scale)  # 使用指定标准差初始化权重。
            torch.nn.init.zeros_(layer.bias)  # 使用零偏置。

    def activate(self, tensor):  # 应用选择的非线性。
        return torch.sigmoid(tensor) if self.activation == "sigmoid" else torch.relu(tensor)  # 返回 Sigmoid 或 ReLU。

    def forward(self, tensor):  # 执行前向并返回全部层激活。
        activations = []  # 保存用于诊断的中间张量。
        hidden = self.activate(self.input_layer(tensor))  # 计算第一层激活。
        hidden.retain_grad()  # 要求反向后保留非叶子激活梯度。
        activations.append(hidden)  # 保存第一层。
        for layer in self.hidden_layers:  # 依次执行九个隐藏层。
            transformed = self.activate(layer(hidden))  # 计算当前非线性变换。
            hidden = hidden + transformed if self.residual else transformed  # 可选加入恒等残差路径。
            hidden.retain_grad()  # 保留当前层输出梯度。
            activations.append(hidden)  # 保存当前层。
        return self.output_layer(hidden), activations  # 返回 Logit 和十层激活。

def diagnose(model):  # 对单次前向反向收集逐层统计。
    logits, activations = model(features)  # 执行模型并保留激活列表。
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels)  # 计算稳定二分类损失。
    model.zero_grad()  # 清空已有参数梯度。
    loss.backward()  # 反向传播到所有层。
    rows = []  # 收集逐层激活和梯度范数。
    for index, activation in enumerate(activations, start=1):  # 从输入侧到输出侧遍历层。
        rows.append({"layer": index, "activation_mean": activation.detach().mean().item(), "activation_std": activation.detach().std(unbiased=False).item(), "gradient_norm": activation.grad.norm().item()})  # 保存统计。
    parameter_gradient = sum(parameter.grad.norm().item() for parameter in model.parameters() if parameter.grad is not None)  # 汇总参数梯度范数。
    return loss.item(), rows, parameter_gradient  # 返回损失、逐层表和总参数梯度。

sigmoid_model = DeepMLP("sigmoid", weight_scale=0.5, residual=False)  # 小增益与 Sigmoid 导数上限在十层链上共同压缩梯度。
sigmoid_loss, sigmoid_rows, sigmoid_parameter_gradient = diagnose(sigmoid_model)  # 运行梯度诊断。
print(f"Sigmoid loss={sigmoid_loss:.4f}，参数梯度和={sigmoid_parameter_gradient:.4e}")  # 输出整体情况。
print("layer  activation_mean  activation_std  gradient_norm")  # 输出逐层表头。
for row in sigmoid_rows:  # 逐层展示从输入到输出梯度。
    print(f"{row['layer']:>2}       {row['activation_mean']:.4f}           {row['activation_std']:.4f}        {row['gradient_norm']:.4e}")  # 展示输入侧梯度极小。

Sigmoid loss=0.8266，参数梯度和=1.2478e+00
layer  activation_mean  activation_std  gradient_norm
 1       0.4986           0.1970        3.2664e-05
 2       0.5276           0.2201        1.2698e-04
 3       0.6086           0.2017        2.8683e-04
 4       0.5693           0.2262        7.6740e-04
 5       0.5562           0.2573        2.0920e-03
 6       0.4714           0.2603        6.6826e-03
 7       0.4772           0.2872        2.0462e-02
 8       0.4260           0.2032        4.8388e-02
 9       0.4906           0.1930        1.2509e-01
10       0.5220           0.1887        2.6083e-01


### 核心实现：He 初始化、ReLU 与残差短路径

In [3]:
he_scale = (2.0 / 16) ** 0.5  # 计算隐藏层 fan_in=16 的 He 标准差近似。
plain_relu_model = DeepMLP("relu", weight_scale=he_scale, residual=False)  # 构造无残差 ReLU 对照。
residual_model = DeepMLP("relu", weight_scale=he_scale * 0.35, residual=True)  # 用较小残差分支避免十次累加爆炸。
plain_loss, plain_rows, plain_parameter_gradient = diagnose(plain_relu_model)  # 诊断普通 ReLU 网络。
residual_loss, residual_rows, residual_parameter_gradient = diagnose(residual_model)  # 诊断残差网络。
print("layer  sigmoid_grad  relu_grad    residual_grad")  # 输出三种网络逐层梯度表头。
for sigmoid_row, plain_row, residual_row in zip(sigmoid_rows, plain_rows, residual_rows):  # 对齐十层统计。
    print(f"{sigmoid_row['layer']:>2}    {sigmoid_row['gradient_norm']:.3e}   {plain_row['gradient_norm']:.3e}   {residual_row['gradient_norm']:.3e}")  # 展示残差输入侧梯度更可达。
sigmoid_ratio = sigmoid_rows[0]["gradient_norm"] / sigmoid_rows[-1]["gradient_norm"]  # 计算 Sigmoid 首层/末层梯度比例。
residual_ratio = residual_rows[0]["gradient_norm"] / residual_rows[-1]["gradient_norm"]  # 计算残差网络比例。
print(f"首层/末层梯度比：Sigmoid={sigmoid_ratio:.3e}，Residual={residual_ratio:.3e}")  # 量化梯度路径改善。

layer  sigmoid_grad  relu_grad    residual_grad
 1    3.266e-05   5.829e-02   1.505e-01
 2    1.270e-04   6.867e-02   1.458e-01
 3    2.868e-04   6.446e-02   1.397e-01
 4    7.674e-04   5.385e-02   1.287e-01
 5    2.092e-03   5.690e-02   1.109e-01
 6    6.683e-03   8.384e-02   9.598e-02
 7    2.046e-02   8.403e-02   8.628e-02
 8    4.839e-02   9.029e-02   8.530e-02
 9    1.251e-01   1.656e-01   7.467e-02
10    2.608e-01   1.881e-01   7.686e-02
首层/末层梯度比：Sigmoid=1.252e-04，Residual=1.958e+00


## 结果解读：过大 ReLU 权重会让激活和梯度爆炸

In [4]:
exploding_model = DeepMLP("relu", weight_scale=2.5, residual=False)  # 构造权重尺度远超 He 的网络。
exploding_loss, exploding_rows, exploding_parameter_gradient = diagnose(exploding_model)  # 收集爆炸统计。
print("layer  activation_std   gradient_norm")  # 输出爆炸网络表头。
for row in exploding_rows:  # 逐层展示激活尺度。
    print(f"{row['layer']:>2}      {row['activation_std']:.3e}      {row['gradient_norm']:.3e}")  # 展示向深层指数放大。
max_activation_std = max(row["activation_std"] for row in exploding_rows)  # 找到最大激活标准差。
max_gradient_norm = max(row["gradient_norm"] for row in exploding_rows)  # 找到最大中间梯度范数。
print(f"Exploding loss={exploding_loss:.3e}，最大激活std={max_activation_std:.3e}，最大激活梯度={max_gradient_norm:.3e}，参数梯度和={exploding_parameter_gradient:.3e}")  # 汇总爆炸程度。
print("解读：诊断要同时看激活和梯度；只有梯度大不一定是问题，但随深度呈数量级增长通常说明初始化或结构不稳。")  # 解释读表方法。

layer  activation_std   gradient_norm
 1      2.334e+00      6.886e+06
 2      1.042e+01      1.168e+06
 3      6.139e+01      3.753e+05
 4      5.717e+02      5.060e+04
 5      4.127e+03      9.347e+03
 6      2.913e+04      1.116e+03
 7      2.449e+05      2.356e+02
 8      2.524e+06      3.070e+01
 9      9.262e+06      1.234e+01
10      5.025e+07      1.684e+00
Exploding loss=4.460e+07，最大激活std=5.025e+07，最大激活梯度=6.886e+06，参数梯度和=2.601e+08
解读：诊断要同时看激活和梯度；只有梯度大不一定是问题，但随深度呈数量级增长通常说明初始化或结构不稳。


## 失败案例：NaN 产生后再 Clip 已经来不及

In [5]:
bad_parameter = torch.nn.Parameter(torch.tensor([1000.0]))  # 构造会进入指数溢出的参数。
bad_loss = torch.exp(bad_parameter).sum()  # 在 FP32 中直接产生 Inf 损失。
bad_loss.backward()  # 反向传播产生 Inf 梯度。
before_clip = bad_parameter.grad.clone()  # 保存裁剪前非有限梯度。
clip_result = torch.nn.utils.clip_grad_norm_([bad_parameter], max_norm=1.0)  # 尝试对 Inf 梯度做范数裁剪。
after_clip = bad_parameter.grad.clone()  # 保存裁剪后结果，通常变成 NaN。
finite_before_step = torch.isfinite(after_clip).all().item()  # 检查优化器更新前梯度是否可用。
print(f"bad_loss={bad_loss.item()}，clip返回范数={clip_result.item()}，裁剪前={before_clip.tolist()}，裁剪后={after_clip.tolist()}，finite={finite_before_step}")  # 展示 clipping 不是 NaN 修复器。
recovery_action = "skip-step-and-reduce-loss-scale" if not finite_before_step else "optimizer-step"  # 根据非有限检测选择恢复动作。
print("恢复动作：", recovery_action)  # 展示必须在 optimizer.step 前跳过更新。
print("生产边界：训练监控应记录每层梯度分位数、update/weight ratio、AMP scale、非有限次数，并保存能重放故障 batch 的 checkpoint 与输入 ID。")  # 总结工程边界。

bad_loss=inf，clip返回范数=inf，裁剪前=[inf]，裁剪后=[nan]，finite=False
恢复动作： skip-step-and-reduce-loss-scale
生产边界：训练监控应记录每层梯度分位数、update/weight ratio、AMP scale、非有限次数，并保存能重放故障 batch 的 checkpoint 与输入 ID。


## 回归测试：最后只保护消失、爆炸、残差与非有限门禁

In [6]:
assert sigmoid_ratio < 1e-3  # 验证十层饱和 Sigmoid 的首层梯度相对末层显著消失。
assert residual_ratio > sigmoid_ratio * 100  # 验证残差路径把首层相对梯度提高至少两个数量级。
assert max_activation_std > 1e6 and max_gradient_norm > 1e5  # 验证过大 ReLU 初始化产生真实爆炸。
assert torch.isinf(bad_loss) and not finite_before_step  # 验证非有限损失与梯度探针确实发生。
assert recovery_action == "skip-step-and-reduce-loss-scale"  # 验证 NaN 后不会继续 optimizer step。
print("回归测试通过：Sigmoid消失、残差改善、ReLU爆炸、Clip失效和非有限跳步均成立。")  # 用少量断言总结梯度诊断合同。

回归测试通过：Sigmoid消失、残差改善、ReLU爆炸、Clip失效和非有限跳步均成立。
